In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# --- Reload data & reproduce train/test split (same as notebook 03) ---
df = pd.read_csv('data/processed/churn_engineered.csv')
df = df.select_dtypes(include='number')
X = df.drop('Churn', axis=1)
y = df['Churn']

imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X_smote, y_smote, test_size=0.2, random_state=42, stratify=y_smote
)

# --- Reload all trained models from pickle ---
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

with open('models/xgb_model.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

# Retrain the other 3 models (only XGBoost was saved)
lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42).fit(X_train, y_train)
lgbm = lgb.LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1).fit(X_train, y_train)

trained_models = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb_model,
    'LightGBM': lgbm
}

# --- 1. Evaluate all models ---
results = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_pred_proba)
    }
    print(f"\n{name}:")
    for k, v in results[name].items():
        print(f"  {k}: {v:.4f}")

# --- 2. Compare models ---
results_df = pd.DataFrame(results).T
print("\n=== Model Comparison ===")
print(results_df.to_string())
best = results_df['ROC-AUC'].idxmax()
print(f"\n\u2705 Best model by ROC-AUC: {best} ({results_df.loc[best, 'ROC-AUC']:.4f})")

# --- 3. ROC curve for best model ---
fpr, tpr, _ = roc_curve(y_test, trained_models[best].predict_proba(X_test)[:, 1])
auc_score = results_df.loc[best, 'ROC-AUC']
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'{best} ROC (AUC={auc_score:.3f})', color='darkorange')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig('models/roc_curve.png', dpi=100)
plt.show()
print('ROC curve saved to models/roc_curve.png')

# --- 4. Confusion matrix ---
cm = confusion_matrix(y_test, trained_models[best].predict(X_test))
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.title(f'Confusion Matrix - {best}')
plt.tight_layout()
plt.savefig('models/confusion_matrix.png', dpi=100)
plt.show()
print('Confusion matrix saved to models/confusion_matrix.png')



Logistic Regression:
  Accuracy: 0.7710
  Precision: 0.7548
  Recall: 0.8029
  F1: 0.7781
  ROC-AUC: 0.8528

Random Forest:
  Accuracy: 0.8242
  Precision: 0.7915
  Recall: 0.8802
  F1: 0.8335
  ROC-AUC: 0.8969

XGBoost:
  Accuracy: 0.8411
  Precision: 0.8186
  Recall: 0.8763
  F1: 0.8465
  ROC-AUC: 0.9236

LightGBM:
  Accuracy: 0.8348
  Precision: 0.8130
  Recall: 0.8696
  F1: 0.8403
  ROC-AUC: 0.9246

=== Model Comparison ===
                     Accuracy  Precision    Recall        F1   ROC-AUC
Logistic Regression  0.771014   0.754768  0.802899  0.778090  0.852789
Random Forest        0.824155   0.791486  0.880193  0.833486  0.896862
XGBoost              0.841063   0.818592  0.876329  0.846477  0.923646
LightGBM             0.834783   0.813008  0.869565  0.840336  0.924575

✅ Best model by ROC-AUC: LightGBM (0.9246)


ROC curve saved to models/roc_curve.png
Confusion matrix saved to models/confusion_matrix.png


/var/folders/j5/js63zckd0rq1nfv2zw1fz7ph0000gn/T/ipykernel_95597/2780257579.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/j5/js63zckd0rq1nfv2zw1fz7ph0000gn/T/ipykernel_95597/2780257579.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
